# VRTX benchmarks on a Kaggle T4 (Months 2 to 4)

Settings (right sidebar): **Accelerator = GPU T4** (T4 x2 is fine, one GPU is used) and **Internet = On**.

Then **Run All**. Takes roughly 45 to 75 minutes. When it finishes:
1. Download `vrtx_results.zip` from the Output panel (all raw JSON + charts), **or**
2. copy the summary printed by the last cell and paste it back.

## 1. GPU, code, requirements

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
%cd /kaggle/working
!rm -rf vrtx
!git clone --depth 1 https://github.com/JacobBhatt12/vrtx.git
%cd vrtx
!git log --oneline -1

In [ ]:
!pip install -q -r requirements.txt
import torch, transformers, triton
print('torch', torch.__version__, '| transformers', transformers.__version__, '| triton', triton.__version__)
assert torch.cuda.is_available(), 'No GPU: set Accelerator to GPU T4 in the sidebar, then restart'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
from huggingface_hub import snapshot_download
for repo, d in [('Qwen/Qwen2.5-0.5B-Instruct', 'models/qwen2.5-0.5b'), ('Qwen/Qwen2.5-1.5B-Instruct', 'models/qwen2.5-1.5b')]:
    snapshot_download(repo, local_dir=d, allow_patterns=['*.json', '*.safetensors', '*.txt'])
!du -sh models/*

## 2. Tests on the real GPU

Includes every Triton kernel against its PyTorch twin in fp16, and the full model with all kernels on.

In [ ]:
!python -m pytest tests/ -q

## 3. Month 2: continuous batching with the PyTorch ops (batched decode attention), vs naive Hugging Face

In [ ]:
!python -m benchmarks.bench_batching --requests 64 --max-tokens 128 --batches 1,8,32,64 --kernels none --hf-baseline

## 4. Month 3: Triton kernels and quantization

In [ ]:
# same workload, Triton kernels on: first without, then with CUDA graphs (the default on CUDA)
!python -m benchmarks.bench_batching --requests 64 --max-tokens 128 --batches 1,8,32,64 --cuda-graphs off
!python -m benchmarks.bench_batching --requests 64 --max-tokens 128 --batches 1,8,32,64

In [ ]:
# int8 weights: autotuned Triton matmul, routed per shape against dequantize + cuBLAS
!python -m benchmarks.bench_batching --requests 64 --max-tokens 128 --batches 1,8,32,64 --quant 8

In [ ]:
# each kernel vs PyTorch (with FLOPs/bytes for the roofline) + kernels switched on one at a time, then CUDA graphs
!python -m benchmarks.bench_kernels

In [ ]:
# perplexity on WikiText-2 vs memory vs speed: fp16, int8, int4 (group 128 / 64 / 32)
!python -m benchmarks.bench_quant

## 5. Month 4: prefix caching, speculative decoding, full serving benchmark

In [ ]:
!python -m benchmarks.bench_prefix

In [ ]:
# target Qwen2.5-1.5B, draft Qwen2.5-0.5B, CUDA graphs on (default)
!python -m benchmarks.bench_spec --target models/qwen2.5-1.5b --draft models/qwen2.5-0.5b

In [ ]:
!python -m benchmarks.bench_spec --target models/qwen2.5-1.5b --draft models/qwen2.5-0.5b --temperature 0.7 --gammas 0,2,4 --batches 1

In [ ]:
!python -m benchmarks.bench_serving --engine vrtx --prefix-caching

In [ ]:
!python -m benchmarks.bench_serving --engine hf --workloads short_chat,shared_prefix --requests 16

## 6. vLLM on the same GPU and prompts

vLLM gets its own virtual environment. Installed into this notebook's Python, it swaps PyTorch for a different CUDA build, the preinstalled TorchAudio no longer matches, and `transformers` fails on import. It runs last anyway: if it fails on the T4, everything above is still saved.

In [ ]:
%cd /kaggle/working/vrtx
# --python names the target explicitly: Kaggle's environment can make uv ignore VIRTUAL_ENV
# and install into the system Python, leaving this venv without torch.
!pip install -q uv
!uv venv -q --python 3.12 /kaggle/working/vllm-env
!uv pip install --python /kaggle/working/vllm-env/bin/python vllm -r requirements.txt 2>&1 | tail -15
!/kaggle/working/vllm-env/bin/python -c "import torch, vllm; print('vllm', vllm.__version__, '| torch', torch.__version__, torch.version.cuda)"
!/kaggle/working/vllm-env/bin/python -m benchmarks.bench_serving --engine vllm --prefix-caching 2>&1 | tail -40 || true
!ls benchmarks/results | grep vllm || echo 'vLLM run failed: the 40 lines above say why'

## 7. Charts and results

In [ ]:
%cd /kaggle/working/vrtx
!python -m benchmarks.plots
!ls docs/charts
!zip -qr /kaggle/working/vrtx_results.zip benchmarks/results docs/charts && echo 'download vrtx_results.zip from the Output panel'

In [ ]:
import json, glob, os
# Compact summary: newest CUDA file of each kind, without the per-request latency lists.
def strip(o):
    if isinstance(o, dict):
        return {k: strip(v) for k, v in o.items() if not k.endswith('_all_s')}
    if isinstance(o, list):
        return [strip(v) for v in o]
    return o
import subprocess
old = set(subprocess.run(['git', 'ls-files', 'benchmarks/results'], capture_output=True, text=True).stdout.split())
newest = {}
for f in sorted(glob.glob('benchmarks/results/*_cuda_*.json'), key=os.path.getmtime):
    if f in old:  # results already in the repo
        continue
    kind = os.path.basename(f).split('_cuda_')[0]
    data = json.load(open(f))
    if kind == 'batching':
        kind += '_' + ('triton' if data.get('kernels') else 'pytorch') + (f"_int{data['quant']}" if data.get('quant') else '') + ('_graphs' if data.get('cuda_graphs') else '')
    newest.setdefault(kind, []).append(strip(data))
print(json.dumps(newest, separators=(',', ':')))